* To start out, please put this notebook into the notebooks folder of the project
  

# Imports

In [100]:
# Imports
import sys
import os

# Add the upstream directory to sys.path
upstream_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if upstream_dir not in sys.path:
    sys.path.insert(0, upstream_dir)

# Now you can import the module
from opentrons_api import ot2_api
from microtissue_manipulator import core, utils, camera
import numpy as np 
import cv2
import time
import json
import keyboard
# from pynput import keyboard
import paths
import matplotlib.pyplot as plt
import requests
import datetime
import threading
import queue
import string
import pandas as pd
import multiprocessing as mp
from dataclasses import dataclass, fields, asdict, MISSING
from typing import get_type_hints, get_origin, get_args, Tuple, List, Dict, Any, Union
from ultralytics import YOLO
from enum import Enum
# from typeguard import typechecked
# from typeguard import install_import_hook
# install_import_hook('__main__')

# Robot initialization

* Clean the robot with 70% Ethanol.
* Remove any objects and platforms from the deck if needed, since the well plate would use the standard deck slot.
* Use the following 4 commands in sequence to set the robot up after it is first turned on.
   
* General information on the jupyter notebook system - the cells can be executed individually, and not necessarily in order, making this proto-gui incredibly flexible.

In [101]:
# 1. Connect the robot to the computer and this notebook
openapi = ot2_api.OpentronsAPI()

In [3]:
# 2. Set the offset if the platform is used
openapi.add_slot_offsets([5,8,9], (0,0,64.2))

In [5]:
# 3. Use the light control to see if the robot is connected as a sanity check. You can leave the light on or turn it off.
openapi.toggle_lights()

<Response [200]>

In [6]:
# 4. Always do once after robot was just turned on
openapi.home_robot()

Request status:
<Response [200]>
{
  "message": "Homing robot."
}


<Response [200]>

In [7]:
# 5. Do after first launch
openapi.create_run()

Request status:
<Response [201]>
{
  "data": {
    "id": "4d39b41c-078a-4967-afee-ad46312049fd",
    "ok": true,
    "createdAt": "2025-06-05T02:10:50.262695Z",
    "status": "idle",
    "current": true,
    "actions": [],
    "errors": [],
    "hasEverEnteredErrorRecovery": false,
    "pipettes": [],
    "modules": [],
    "labware": [],
    "liquids": [],
    "liquidClasses": [],
    "labwareOffsets": [],
    "runTimeParameters": [],
    "outputFileIds": []
  }
}


<Response [201]>

In [8]:
# 6. Let the robot know that it has the P300 pipette
openapi.load_pipette()

Request status:
<Response [201]>
{
  "data": {
    "id": "0f74c188-b110-4b52-84b7-9dd30f15d283",
    "createdAt": "2025-06-05T02:10:55.429268Z",
    "commandType": "loadPipette",
    "key": "0f74c188-b110-4b52-84b7-9dd30f15d283",
    "status": "succeeded",
    "params": {
      "pipetteName": "p300_single_gen2",
      "mount": "left"
    },
    "result": {
      "pipetteId": "a3f8ec49-cc85-48d6-ace1-15238d889594"
    },
    "startedAt": "2025-06-05T02:10:55.432570Z",
    "completedAt": "2025-06-05T02:10:57.384968Z",
    "intent": "setup",
    "notes": []
  }
}


<Response [201]>

* If a crash ocurred at any point, and you need to restart the notebook, run commands 1,2,3 again, but NOT 4,5,6. Instead, execute command 7 in the following code cell. 

In [ ]:
# 7. Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

* At this point, the robot setup is complete, and it's time to put some labware on the deck.

# Labware declaration

### Opentrons 300ul

* Locate some sterile 300ul tips on the shelves to the right of the opentrons. It is a black box with semi-transparent lid with orange tape. 
* Insert the box into slot number 11. 

In [9]:
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "opentrons_96_tiprack_300ul"
#Load the tip rack. Slot = 1 by default.
r = openapi.load_labware(TIP_RACK, 10)

Labware ID:
324818a8-368d-4d5c-b702-3eb791131783



* The following command is used to pick up the tip. Input the proper coordinate for the tip: 'A1', 'B2', etc.

In [97]:
r = openapi.pick_up_tip(openapi.labware_dct['10'], "A7")

### Reservoirs and well plates

* Install a 6 well plate in slot 4. This is where the extracted solution will be dumped.

In [11]:
# Define a reservoir or well plate to dump the extracted solution
RESERVOIR = "corning_6_wellplate_16.8ml_flat"
r = openapi.load_labware(RESERVOIR, 4, namespace='opentrons',verbose=True)

Labware ID:
5befb3d6-4ccf-4b88-8ef1-e76e5a8f10a0



* Install the well plate the will be used in slot 1.

In [39]:
# Define the well plate that will be used

WELL_PLATE = "corning_96_wellplate_360ul_flat"
# WELL_PLATE = "corning_24_wellplate_3.4ml_flat"
#WELL_PLATE = "corning_384_wellplate_112ul_flat"
r = openapi.load_labware(WELL_PLATE, 1, namespace='opentrons',verbose=True)

Labware ID:
fb51e2db-56b5-445e-a536-aec8642bb16c



* If you will be removing solution from multiple well plates, and their type is different, first, remove the old well plate with the following command. Then proceed to executing the previous 2 cells again with the proper well type selected.

In [38]:
# Move the well plate FROM SLOT 1 off the deck if you want to change it
openapi.move_labware(openapi.labware_dct['1'], new_location='offDeck')

<Response [201]>

# Removing solution

* At this point all of the labware that is used should be installed in the slots of the deck.
* A pipette tip should be installed.
* There will be a small calibration procedure that needs to be done before removing solution.

In [64]:
# Toggle on the light so you can see better if not already.
openapi.toggle_lights()

<Response [200]>

* The following command moves the pipette tip over a single well. Make sure that the tip is centered over the well. You can do that by changing the x and y offset values and then launching the cell again. 

In [65]:
# Center the pipette tip over a well in the well plate. This is a calibration step that needs to be done before removing solution.
x_offset = 0
y_offset = 0
z_offset = 1
openapi.move_to_well(openapi.labware_dct['1'], "H1", well_location='top', offset=(x_offset,y_offset,z_offset))

<Response [201]>

* After the tip is centered over the well, it is time to determine the well bottom height.
* To do that, the manual movement is activated, and the tip is touched down to the side border of the well - the top rim.
* Once the position is recorded, subtract the well height to find the well bottom.
* Don't forget to turn off manual mode after you are done with it.

In [66]:
# Activate the manual movement mode.
# Controls: arrow keys for XY movement, PageUp PageDown for Z movement.
# Press + or - to change the movement step size. Starting step size is 1mm. Next one down 0.5 then 0.1mm.
# Press 's' to save the current position of the robot.
manual_movement = utils.ManualRobotMovement(openapi)

In [67]:
# Use this command to turn off manual movement.
keyboard.unhook_all()

In [68]:
# You can also use this command to retract the pipette tip to the very top.
openapi.retract_axis('leftZ')

<Response [201]>

In [69]:
# Execute this command to see which position was saved. Take the Z coordinate - the last number.
manual_movement.positions

[]

* Now that you have the Z coordinate of the top of the well plate, subtract the height of the well to get the bottom of the well plate.

Well heights for different plate types:
* 12.5 - 384U
* 10.8 - 96U
* 11.5 - 384Flat
* 10.5 - 96Flat Clear
* 11.0 - 96Flat Black

In [70]:
# Execute this cell to calculate the well bottom height.
#  For example, if plate top z = 14.52, and the plate is 384-U-bottom:
well_plate_bottom = 14.72 - 11.0 # 14.72 is plate top, 11.0 is well height
well_plate_bottom

3.7200000000000006

* It is time to determine the well locations:
* A well plate preview will appear, be sure to check that everything is correct.

In [98]:
plate_type = 96 # Use proper well type.
dest = core.Destination(plate_type)
well_df = core.create_well_plan(plate_type)

# Use this line to define which wells to take solution from.
# You can define the range of wells and columns by using the following syntax:
# If you want rows from B to H and columns from 2 to 11, use:
well_df.loc['B':'C', 7:7] = 1

well_plan = {f"{row}{col}": well_df.loc[row, col] for row in well_df.index for col in well_df.columns if well_df.loc[row, col] > 0}
take_off_solution_routine = core.Routine(dest, well_plan)
well_df

,1,2,3,4,5,6,7,8,9,10,11,12
A,0,0,0,0,0,0,0,0,0,0,0,0
B,0,0,0,0,0,0,1,0,0,0,0,0
C,0,0,0,0,0,0,1,0,0,0,0,0
D,0,0,0,0,0,0,0,0,0,0,0,0
E,0,0,0,0,0,0,0,0,0,0,0,0
F,0,0,0,0,0,0,0,0,0,0,0,0
G,0,0,0,0,0,0,0,0,0,0,0,0
H,0,0,0,0,0,0,0,0,0,0,0,0


* Once you picked the proper wells, time to remove solution finally

In [99]:
z_offset = 1
aspirate_z_offset = 3.2 # Need to determine this based on how much liquid you want to leave behind.
takeoff_volume = 120 # Estimate this on how much liquid is in the well.
flow_rate = 20 # If high up above the bottom, can use higher flow rate like 50. Otherwise, 10ul/s is good but slow.
well_plate_slot = '1'
reservoir_slot = '4'

openapi.retract_axis('leftZ')
openapi.blow_out(openapi.labware_dct[reservoir_slot], "A2", well_location='top', flow_rate = 200)
openapi.aspirate(openapi.labware_dct[reservoir_slot], "A1", well_location='top', offset = (0,0, z_offset), volume = 10, flow_rate = 200)
openapi.dispense(openapi.labware_dct[reservoir_slot], "A1", well_location='top', offset = (0,0, z_offset), volume = 10, flow_rate = 200)
openapi.retract_axis('leftZ')

while not take_off_solution_routine.is_done():
    well = take_off_solution_routine.get_next_well()
    openapi.move_to_well(openapi.labware_dct[well_plate_slot], well, well_location='top', offset=(x_offset, y_offset, z_offset))
    robot_x, robot_y, robot_z = openapi.get_position(verbose=False)[0].values()
    openapi.move_to_coordinates((robot_x, robot_y, well_plate_bottom + aspirate_z_offset), min_z_height=well_plate_bottom, verbose=False, force_direct=True)
    r = openapi.aspirate_in_place(volume = takeoff_volume, flow_rate = flow_rate)

    openapi.move_to_well(openapi.labware_dct[well_plate_slot], well, well_location='top', offset=(x_offset, y_offset, z_offset), force_direct=True)
    responce_dict = json.loads(r.text)['data']
    if responce_dict['status'] == 'failed':
        if responce_dict['error']['errorType'] == 'InvalidAspirateVolumeError':
            print('Dumping fluid')
            openapi.blow_out(openapi.labware_dct[reservoir_slot], "A2", well_location='top', flow_rate = 200)
            # time.sleep(5)
            openapi.aspirate(openapi.labware_dct[reservoir_slot], "A1", well_location='top', offset = (0,0, z_offset), volume = 10, flow_rate = 200)
            openapi.dispense(openapi.labware_dct[reservoir_slot], "A1", well_location='top', offset = (0,0, z_offset), volume = 10, flow_rate = 200)
    else:
        take_off_solution_routine.update_well(success=True)
        take_off_solution_routine.get_next_well()

    
openapi.retract_axis('leftZ')
openapi.blow_out(openapi.labware_dct[reservoir_slot], "A2", well_location='top', flow_rate = 200)
openapi.aspirate(openapi.labware_dct[reservoir_slot], "A1", well_location='top', offset = (0,0, z_offset), volume = 10, flow_rate = 200)
openapi.dispense(openapi.labware_dct[reservoir_slot], "A1", well_location='top', offset = (0,0, z_offset), volume = 10, flow_rate = 200)
openapi.retract_axis('leftZ')

<Response [201]>

* If you want to remove the tip, I recommend just taking it off manually and then letting the robot know that there's no tip anymore.
* After removing tip, scroll up the notebook to find labware declaration.
* Attach a new tip, and repeat at least the XY calibration process (centering the tip to the well).
* Z calibration can be skipped since experinece shows tips are more different in XY than Z.

In [96]:
openapi.drop_tip_in_place()

<Response [201]>

* You can repeat taking off solution from a different set of wells, so you will need to re-execute some of the code above. 